In [1]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [2]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        # ---- assemble row (explicit fields)
        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    # ---- FORCE column order ----
    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]


# Chạy với các version data

## V1 (Median)

In [3]:
path_1 = "/kaggle/input/lo-dataset/Median/Median"

In [4]:
path_2 = "/kaggle/input/rf-result/RF_result"

In [5]:
df_v1 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V1 (Median)",
    train_name="train_median"
)
df_v1

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median,V1 (Median),1,1,0.000129,0.998467,0.545811,0.998467,0.5,0.804970,1.0
1,train_median,V1 (Median),2,1,0.000987,0.999206,0.671268,0.999206,0.5,0.833418,1.0
2,train_median,V1 (Median),3,1,0.002904,0.999817,0.585785,0.999817,0.5,0.814876,1.0
3,train_median,V1 (Median),4,1,0.004065,0.999952,0.000089,0.999952,0.5,0.188133,1.0


In [6]:
df_v1.to_csv("results_v1.csv", index=False)

## V2 (Median CDSMOTE)

In [7]:
df_v2 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V2 (Median CDS)",
    train_name="train_median_cdsmote"
)
df_v2

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_cdsmote,V2 (Median CDS),1,1,0.001484,0.999248,0.436394,0.545515,0.5,0.701275,1.0
1,train_median_cdsmote,V2 (Median CDS),2,1,0.001607,0.999406,0.645152,0.546032,0.5,0.748627,1.0
2,train_median_cdsmote,V2 (Median CDS),3,1,0.006898,0.999928,0.728637,0.559592,0.5,0.767162,1.0
3,train_median_cdsmote,V2 (Median CDS),4,1,0.007821,0.999885,0.000000,0.561442,0.5,0.017433,1.0


In [8]:
df_v2.to_csv("results_v2.csv", index=False)

## V3 (Median SASMOTE)

In [9]:
df_v3 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V3 (Median SAS)",
    train_name="train_median_sasmote"
)
df_v3

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_sasmote,V3 (Median SAS),1,1,0.000891,0.999027,0.491068,0.543868,0.5,0.714821,1.0
1,train_median_sasmote,V3 (Median SAS),2,1,0.001426,0.999328,0.675560,0.545505,0.5,0.754264,1.0
2,train_median_sasmote,V3 (Median SAS),3,1,0.006627,0.999933,0.748936,0.559012,0.5,0.770551,1.0
3,train_median_sasmote,V3 (Median SAS),4,1,0.007776,0.999890,0.000000,0.561333,0.5,0.017432,1.0


In [10]:
df_v3.to_csv("results_v3.csv", index=False)

## V4 (Median RadiusSMOTE)

In [11]:
df_v4 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V4 (Median Radius)",
    train_name="train_median_radiussmote"
)
df_v4

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_radiussmote,V4 (Median Radius),1,1,0.000000,0.998217,0.426308,0.540852,0.5,0.697428,1.0
1,train_median_radiussmote,V4 (Median Radius),2,1,0.000123,0.998455,0.585463,0.541424,0.5,0.735455,1.0
2,train_median_radiussmote,V4 (Median Radius),3,1,0.002265,0.999686,0.773208,0.548190,0.5,0.772106,1.0
3,train_median_radiussmote,V4 (Median Radius),4,1,0.004891,0.999981,0.000009,0.554923,0.5,0.115448,1.0


In [12]:
df_v4.to_csv("results_v4.csv", index=False)

## V5 (Mean)

In [13]:
path_1 = "/kaggle/input/lo-dataset/Mean/Mean"

In [14]:
df_v5 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V5 (Mean)",
    train_name="train_mean"
)
df_v5

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean,V5 (Mean),1,1,0.000000,0.998217,0.466757,0.998217,0.5,0.784185,1.0
1,train_mean,V5 (Mean),2,1,0.000387,0.998769,0.534615,0.998770,0.5,0.802276,1.0
2,train_mean,V5 (Mean),3,1,0.002807,0.999805,0.489547,0.999805,0.5,0.790859,1.0
3,train_mean,V5 (Mean),4,1,0.004130,0.999958,0.000005,0.999958,0.5,0.117136,1.0


In [15]:
df_v5.to_csv("results_v5.csv", index=False)

## V6 (Mean CDSMOTE)

In [16]:
df_v6 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V6 (Mean CDS)",
    train_name="train_mean_cdsmote"
)
df_v6

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_cdsmote,V6 (Mean CDS),1,1,0.003291,0.999507,0.212148,0.549934,0.5,0.622708,1.0
1,train_mean_cdsmote,V6 (Mean CDS),2,1,0.009937,0.998883,0.397911,0.563050,0.5,0.694176,1.0
2,train_mean_cdsmote,V6 (Mean CDS),3,1,0.014687,0.998898,0.397797,0.574725,0.5,0.696523,1.0
3,train_mean_cdsmote,V6 (Mean CDS),4,1,0.007776,0.999889,0.000293,0.561336,0.5,0.208575,1.0


In [17]:
df_v6.to_csv("results_v6.csv", index=False)

## V7 (Mean SASMOTE)

In [18]:
df_v7 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V7 (Mean SAS)",
    train_name="train_mean_sasmote"
)
df_v7

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_sasmote,V7 (Mean SAS),1,1,0.005337,0.999467,0.274343,0.554344,0.5,0.650832,1.0
1,train_mean_sasmote,V7 (Mean SAS),2,1,0.004865,0.999513,0.403268,0.553392,0.5,0.693795,1.0
2,train_mean_sasmote,V7 (Mean SAS),3,1,0.006885,0.999931,0.477415,0.559553,0.5,0.714956,1.0
3,train_mean_sasmote,V7 (Mean SAS),4,1,0.007827,0.999885,0.000145,0.561457,0.5,0.185476,1.0


In [19]:
df_v7.to_csv("results_v7.csv", index=False)

## V8 (Mean RadiusSMOTE)

In [20]:
df_v8 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V8 (Mean Radius)",
    train_name="train_mean_radiussmote"
)
df_v8

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_radiussmote,V8 (Mean Radius),1,1,0.000361,0.998692,0.334873,0.542233,0.5,0.670262,1.0
1,train_mean_radiussmote,V8 (Mean Radius),2,1,0.004730,0.999533,0.343282,0.553126,0.5,0.675368,1.0
2,train_mean_radiussmote,V8 (Mean Radius),3,1,0.002678,0.999475,0.483342,0.549309,0.5,0.714171,1.0
3,train_mean_radiussmote,V8 (Mean Radius),4,1,0.005485,0.999995,0.000000,0.556270,0.5,0.017406,1.0


In [21]:
df_v8.to_csv("results_v8.csv", index=False)

## V9 (Extra Trees)

In [22]:
path_1 = "/kaggle/input/lo-dataset/Extra_trees/Extra_trees"

In [23]:
df_v9 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V9 (Extra)",
    train_name="train_extra"
)
df_v9

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra,V9 (Extra),1,1,0.000000,0.998217,0.519205,0.998217,0.5,0.798227,1.0
1,train_extra,V9 (Extra),2,1,0.000019,0.998260,0.619803,0.998260,0.5,0.822151,1.0
2,train_extra,V9 (Extra),3,1,0.001400,0.999407,0.615196,0.999408,0.5,0.821444,1.0
3,train_extra,V9 (Extra),4,1,0.004227,0.999962,0.000029,0.999962,0.5,0.155953,1.0


In [24]:
df_v9.to_csv("results_v9.csv", index=False)

## V10 (Extra Trees CDSMOTE)

In [25]:
df_v10 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V10 (Extra CDS)",
    train_name="train_extra_cdsmote"
)
df_v10

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_cdsmote,V10 (Extra CDS),1,1,0.000000,0.998217,0.527318,0.540852,0.5,0.722588,1.0
1,train_extra_cdsmote,V10 (Extra CDS),2,1,0.000155,0.998481,0.572178,0.541507,0.5,0.732668,1.0
2,train_extra_cdsmote,V10 (Extra CDS),3,1,0.004246,0.999859,0.625748,0.553461,0.5,0.746562,1.0
3,train_extra_cdsmote,V10 (Extra CDS),4,1,0.006795,0.999956,0.000000,0.559203,0.5,0.017422,1.0


In [26]:
df_v10.to_csv("results_v10.csv", index=False)

## V11 (Extra Trees SASMOTE)

In [27]:
df_v11 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V11 (Extra SAS)",
    train_name="train_extra_sasmote"
)
df_v11

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_sasmote,V11 (Extra SAS),1,1,0.000000,0.998217,0.544701,0.540852,0.5,0.726505,1.0
1,train_extra_sasmote,V11 (Extra SAS),2,1,0.000019,0.998269,0.641249,0.540953,0.5,0.746564,1.0
2,train_extra_sasmote,V11 (Extra SAS),3,1,0.002762,0.999707,0.649539,0.549625,0.5,0.750330,1.0
3,train_extra_sasmote,V11 (Extra SAS),4,1,0.006788,0.999957,0.000000,0.559190,0.5,0.017421,1.0


In [28]:
df_v11.to_csv("results_v11.csv", index=False)

## V12 (Extra Trees RadiusSMOTE)

In [29]:
df_v12 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V12 (Extra Radius)",
    train_name="train_extra_radiussmote"
)
df_v12

,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_radiussmote,V12 (Extra Radius),1,1,0.000000,0.998217,0.430757,0.540852,0.5,0.698636,1.0
1,train_extra_radiussmote,V12 (Extra Radius),2,1,0.000484,0.998788,0.496965,0.542633,0.5,0.715944,1.0
2,train_extra_radiussmote,V12 (Extra Radius),3,1,0.000258,0.998634,0.713413,0.541956,0.5,0.760233,1.0
3,train_extra_radiussmote,V12 (Extra Radius),4,1,0.004988,0.999988,0.000000,0.555133,0.5,0.017400,1.0


In [30]:
df_v12.to_csv("results_v12.csv", index=False)